# Does bounded confidence change the controversy-axis result?

An A/B backtest for [Lightningfish](https://github.com/rajul-kk/LightningFish),
run on Kaggle's free T4 GPU — same pattern as `kaggle_controversy.ipynb`
(qwen2.5:7b served locally by Ollama, no API key, $0), sized the same way
(24 agents, 3-4 rounds) since the GPU makes that tractable, unlike the
~4 tokens/sec CPU-only run this started as.

## What's being tested

METHODOLOGY.md's calibrated controversy run found the simulation's crowd-split
prediction lands **below chance** (38% vs. a 53% best baseline, n=74). Since
then, T3's herding update gained a bounded-confidence gate (Hegselmann-Krause
style): an agent ignores a herding target more than `confidence_bound` away
from its own opinion, instead of being pulled toward it. The hypothesis is
that this produces more realistic multi-modal splits than the jitter-dependent
fragmentation the engine relied on before.

This notebook runs the **same calibrated-controversy harness**, same events,
same split logic, twice — gate on (current default) vs. gate forced off
(`confidence_bound=2.0` for everyone, i.e. the exact pre-gate behavior) — and
compares. This is a mechanism test, not a fresh claim that bounded confidence
"works": the honest prior, given every other axis on this domain has failed
the ladder, is that it probably won't move the needle much. Reporting that
plainly either way is the point — see `kaggle_controversy.ipynb` for the same
framing applied to the axis itself.


---
## What the data is

**Source:** the [Hacker News Algolia API](https://hn.algolia.com/api) — free,
unauthenticated, ~10k requests/hour. No key, no scraping.

**Sample:** settled stories at least 24 hours old. `PULL_LIMIT` needs to be
generous — a local run at `limit=60` only cleared 20 scoreable events (33%),
well under the harness's 15/15 minimum split, so this uses 250.

**The seed each agent reads** — strictly submission-time fields, never the
outcome (title, author + karma, url domain, type, self-text). Point-in-time
safety is enforced in code and tested.

**The label:** `num_comments / points` at settlement — ratio >= 0.7 is
"contested," < 0.4 is "consensus," the gap and anything under 20 points is
skipped. See `kaggle_controversy.ipynb`'s data section for the full rationale.


---
## What's different between the two arms

Both arms run `_run_hn_controversy_calibrated` — same event pull, same
calibration/evaluation split (deterministic hash of event id), same
threshold-from-median-of-calibration logic. The **only** difference is
`bounded_confidence`, passed straight to `build_personas`, which controls
whether each HN archetype's `confidence_bound` is applied (on) or forced to
2.0 — the max possible opinion distance, i.e. never gates (off).

Because no local run cache exists yet for this model/size combination, both
arms simulate fresh — this is a real 2x cost over the single-arm
`kaggle_controversy.ipynb`, not a comparison against an already-paid-for run.


---
## 1. Setup

Sidebar: **Accelerator -> GPU** and **Internet -> On**.

`zstd` first: Ollama's installer needs it to extract, Kaggle's image doesn't
ship it, and without this the install fails quietly and surfaces later as a
confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")


In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")


In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest praw yfinance

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms bounded confidence, the
# CachingAdapter kwarg-forwarding fix, and the calibrated-threshold code are
# actually present in this clone (all pushed in commit 17bcc07).
!python -m pytest tests/core tests/hn -q 2>&1 | tail -5


---
## 2. Configuration


In [ ]:
PULL_LIMIT = 250      # stories to pull; expect roughly 1 in 3 to be scoreable
N_AGENTS   = 24        # matches kaggle_controversy.ipynb's validated GPU size
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}, two arms")


### Throughput check

~26 model calls per event, times two arms. Confirm the per-call cost before
committing — if this shows double digits you're on CPU regardless of what the
assert above said.


In [ ]:
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event  ->  ~{per_call*26*2:.0f}s per event-pair (both arms)")


---
## 3. Run both arms

`hn-controversy-calibrated` (gate on) and `hn-controversy-calibrated-nobc`
(gate off) — same event pull and split, only the bounded-confidence flag
differs, so the two reports are directly comparable.


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_on.log


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated-nobc {PULL_LIMIT} 2>&1 | tee /kaggle/working/bc_off.log


### Reading the logs

- `X of Y events have a controversy direction` — how much the points floor and
  gap zone discarded (should match between the two logs — same pull).
- `split: N calibration / M evaluation` — abort if either is under ~15.
- `calibration stddev range: a-b, median (threshold) = t` — the derived
  cutoff; the two arms will generally calibrate to different thresholds since
  bounded confidence changes the simulated distributions.
- The final report block in each log: `beats_baselines` and `p_value_vs_best`.

Compare `sim` accuracy between the two logs:

- **Materially higher with the gate on** — bounded confidence is worth
  carrying forward as a real mechanism.
- **About the same, or worse** — the honest conclusion the rest of this
  project's findings would predict: HN reception is driven by who posts and
  replies early, not by which local opinion-update rule the crowd uses, and
  this axis stays a **fails** either way.

At n≈20-25 per arm (after the 40/60 calibration/evaluation split), individual
accuracy points move by roughly one event each — don't over-read a few points
of difference; look at whether `beats_baselines` and the verdict actually flip.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache


Both logs and the run cache are saved to `/kaggle/working/` — download them
from the notebook's Output tab. The cache holds every simulated run's final
distribution keyed by `bounded_confidence` (`:bc1` suffix vs. none), so a
future question about these same events costs nothing to re-score.

Either result belongs in
[METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md),
alongside the rest — a negative is exactly as reportable as the other rows in
that table.
